# Conservative inflow process-noise tuning

This notebook prepares an actual Aquarius reservoir time window with the repository's canonical `prepare_reservoir_data` workflow, then proposes a prior hourly increment standard deviation for the reservoir inflow random walk. It uses causal, pre-update predictive scores over fixed validation windows and selects the smallest candidate that is practically competitive with the best candidate.

**Safety:** the result is a proposed `ReservoirConfig` only. Nothing is persisted or changed in an operational stream. Keep a contiguous final test period untouched, review warnings and regime diagnostics, and obtain engineering approval before assigning a production configuration version.

In [ ]:
import os
import sys
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)


def find_project_root() -> Path:
    explicit_root = os.environ.get("KALMONE_NOTEBOOK_ROOT")
    candidates = ([Path(explicit_root)] if explicit_root else []) + [
        Path.cwd(), *Path.cwd().parents
    ]
    for candidate in candidates:
        if (candidate / "Reservoirs").is_dir() and (candidate / "src" / "kalmone").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository. Launch from the project or set KALMONE_NOTEBOOK_ROOT."
    )

ROOT = find_project_root()
for import_root in (ROOT / "src", ROOT / "Notebooks"):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from kalmone import (
    InflowUnits,
    InitializationStrategy,
    InflowTuningSettings,
    ReservoirConfig,
    TuningWindow,
    UnitSystem,
    evaluate_inflow_config,
    get_reservoir_inflow_from_config,
    tune_inflow_process_noise,
)

from prepare_reservoir_data import (  # noqa: E402
    prepare_reservoir_data,
    sources_for_reservoir,
)

## Editable parameters

Change values in this cell before a run. The canonical preparer reads the selected Aquarius reservoir and closed `DATA_START`/`DATA_END` interval. Validation windows use half-open intervals: `start <= timestamp < end`; they must fall inside the prepared range and before the reserved test period.

In [ ]:
# ===================== USER PARAMETERS =====================
# These choices are passed to the repository's canonical Aquarius preparer.
PROJECT_ROOT = None  # Optional Path/string override; None discovers the current repository.
RESERVOIR = "Chesbro"  # Supported values are defined by sources_for_reservoir.
DATA_START = pd.Timestamp("2022-11-01T00:00:00Z")
DATA_END = pd.Timestamp("2023-11-01T00:00:00Z")
ASOF_TOLERANCE = pd.Timedelta("20min")

RESERVOIR_ID = RESERVOIR.casefold()
RESERVOIR_NAME = RESERVOIR
MODEL_VERSION = "physical-rate-v1"
BASE_CONFIGURATION_VERSION = "reviewed-base-v1"
PROPOSED_CONFIGURATION_VERSION = "2026-08-q-inflow-candidate"

# These are continuous-time diagonal covariance values. R and P0 are fixed during tuning.
Q_STORAGE = 0.0
Q_OUTFLOW = 0.02
R_STORAGE = 0.25
R_OUTFLOW = 4.0
P0_STORAGE = 100.0
P0_INFLOW = 400.0
P0_OUTFLOW = 25.0
SMOOTHING_LAG = timedelta(hours=12)

# Candidate prior SD values are SDs of one-hour latent inflow increments, not filtered SDs.
CANDIDATE_PRIOR_HOURLY_INCREMENT_SD = np.array([2.0, 5.0, 10.0, 20.0, 40.0])
VALIDATION_REGIMES = (
    ("storm-1", "storm", "1982-11-05T00:00:00Z", "1982-11-10T00:00:00Z"),
    ("dry-1", "dry", "1982-11-12T00:00:00Z", "1982-11-17T00:00:00Z"),
    ("storm-2", "storm", "1982-11-19T00:00:00Z", "1982-11-24T00:00:00Z"),
)
WINDOW_WEIGHTS = None  # Use equal weights; or provide one positive weight per window.
WARMUP = timedelta(hours=24)
INNOVATION_MAX_LAG = timedelta(hours=24)
FORECAST_HORIZONS = (timedelta(hours=1), timedelta(hours=3), timedelta(hours=6))
PRACTICAL_EQUIVALENCE_TOLERANCE = 0.02
BOOTSTRAP_SAMPLES = 1000
RANDOM_SEED = 20260820
MAX_JITTER_FRACTION = 1e-9
MAX_REGULARIZED_STEPS = 0
R_SENSITIVITY_MULTIPLIERS = (0.75, 1.0, 1.25)

# Reserve the final-test period from tuning even when evaluation is disabled.
RESERVE_UNTOUCHED_TEST_PERIOD = True
# This is only an execution toggle; it does not control reservation.
RUN_UNTOUCHED_TEST = False
TEST_START = pd.Timestamp("1982-11-25T00:00:00Z")
TEST_END = pd.Timestamp("1982-12-01T00:00:00Z")
# ============================================================

## Data loading

The canonical `prepare_reservoir_data` function reads the checked-in Aquarius exports under `Reservoirs/<reservoir>/`, performs source-specific parsing and backward as-of alignment, and returns `PreparedReservoirData`. Its `observations` frame has exactly numeric `storage` and `outflow` columns with a timezone-aware, strictly increasing UTC index; `diagnostics`, `source_audit`, and `window_audit` remain available for review.

The configured reservoir, date range, and as-of tolerance are centralized above. The source files are prerequisites; if the repository root, reservoir choice, or exports are unavailable, the preparation cell fails with an actionable message. Missing values may remain `NaN` after preparation, but this notebook does not sort, interpolate, deduplicate, or silently repair the returned model frame.

In [ ]:
try:
    project_root = ROOT if PROJECT_ROOT is None else Path(PROJECT_ROOT).expanduser().resolve()
    sources = sources_for_reservoir(project_root, RESERVOIR)
    prepared = prepare_reservoir_data(
        sources, start=DATA_START, end=DATA_END, asof_tolerance=ASOF_TOLERANCE
    )
except FileNotFoundError as error:
    raise RuntimeError(
        "Reservoir source files are unavailable. Check KALMONE_NOTEBOOK_ROOT, RESERVOIR, and the Aquarius exports under Reservoirs/."
    ) from error
except (ValueError, OSError) as error:
    raise RuntimeError(
        "Could not prepare reservoir data. Check DATA_START/DATA_END, source format, and Aquarius export prerequisites."
    ) from error

observations = prepared.observations
data_source = f"{RESERVOIR}: prepare_reservoir_data"
reserved_test_start = pd.Timestamp(TEST_START)
reserved_test_end = pd.Timestamp(TEST_END)
if reserved_test_end <= reserved_test_start:
    raise ValueError("TEST_END must be after TEST_START")
if RESERVE_UNTOUCHED_TEST_PERIOD:
    if not (observations.index.min() < reserved_test_start <= observations.index.max()):
        raise ValueError("TEST_START must fall inside the prepared date range")
    if reserved_test_end > observations.index.max() + pd.Timedelta(minutes=1):
        raise ValueError("TEST_END extends beyond the prepared date range")
    tuning_mask = observations.index < reserved_test_start
else:
    tuning_mask = np.ones(len(observations), dtype=bool)
if not tuning_mask.any():
    raise ValueError("The reserved test period leaves no observations for tuning")

full_storage = observations["storage"].copy()
full_discharge = observations["outflow"].copy()
tuning_observations = observations.loc[tuning_mask].copy()
storage = tuning_observations["storage"].copy()
discharge = tuning_observations["outflow"].copy()
tuning_storage = storage
tuning_discharge = discharge

display(prepared.source_audit[["path", "source_rows", "clean_rows", "duplicates_removed", "finite_value_fraction", "first_utc", "last_utc"]])
display(prepared.window_audit)
display(tuning_observations.head())
display(tuning_observations.isna().mean().rename("missing_fraction").to_frame())
print(f"Prepared source: {data_source}")
print(f"Prepared rows: {len(observations):,}; range: {observations.index.min()} through {observations.index.max()}")
print(f"Tuning rows: {len(tuning_observations):,}; range: {tuning_observations.index.min()} through {tuning_observations.index.max()}")
if RESERVE_UNTOUCHED_TEST_PERIOD:
    print(f"Reserved untouched test range: {reserved_test_start} <= timestamp < {reserved_test_end}")

In [ ]:
if not storage.index.equals(discharge.index):
    raise ValueError("storage and discharge indexes must match exactly")
if storage.index.tz is None or not storage.index.is_monotonic_increasing or storage.index.has_duplicates:
    raise ValueError("the input index must be timezone-aware and strictly increasing")

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(storage.index, storage, label="storage")
axes[0].set_ylabel("Storage")
axes[0].legend()
axes[1].plot(discharge.index, discharge, label="discharge", color="tab:orange")
axes[1].set_ylabel("Discharge")
axes[1].legend()
fig.tight_layout()
plt.show()

## Build the fixed base configuration and validation windows

In [ ]:
base_config = ReservoirConfig(
    reservoir_id=RESERVOIR_ID,
    reservoir_name=RESERVOIR_NAME,
    q=np.diag([Q_STORAGE, 1.0, Q_OUTFLOW]),
    r=np.diag([R_STORAGE, R_OUTFLOW]),
    p0=np.diag([P0_STORAGE, P0_INFLOW, P0_OUTFLOW]),
    smoothing_lag=SMOOTHING_LAG,
    initialization_strategy=InitializationStrategy.FIRST_TWO_VALID_STORAGE,
    inflow_units=InflowUnits.CUBIC_FEET_PER_SECOND,
    model_version=MODEL_VERSION,
    configuration_version=BASE_CONFIGURATION_VERSION,
    metadata={"source": data_source, "status": "reviewed base"},
    unit_system=UnitSystem.us_customary(),
)

validation_windows = tuple(
    TuningWindow(
        name=name,
        regime=regime,
        start=pd.Timestamp(start),
        end=pd.Timestamp(end),
        weight=None if WINDOW_WEIGHTS is None else WINDOW_WEIGHTS[position],
    )
    for position, (name, regime, start, end) in enumerate(VALIDATION_REGIMES)
)

if RESERVE_UNTOUCHED_TEST_PERIOD:
    overlapping_windows = [
        window.name
        for window in validation_windows
        if window.end > reserved_test_start
    ]
    if overlapping_windows:
        raise ValueError(
            "Validation windows overlap or extend into the reserved test period: "
            + ", ".join(overlapping_windows)
        )

settings = InflowTuningSettings(
    warmup=WARMUP,
    innovation_max_lag=INNOVATION_MAX_LAG,
    forecast_horizons=FORECAST_HORIZONS,
    min_scored_storage_observations=12,
    practical_equivalence_tolerance=PRACTICAL_EQUIVALENCE_TOLERANCE,
    bootstrap_samples=BOOTSTRAP_SAMPLES,
    random_seed=RANDOM_SEED,
    max_jitter_fraction=MAX_JITTER_FRACTION,
    max_regularized_steps=MAX_REGULARIZED_STEPS,
    r_sensitivity_multipliers=R_SENSITIVITY_MULTIPLIERS,
)

display(pd.DataFrame({
    "name": [window.name for window in validation_windows],
    "regime": [window.regime for window in validation_windows],
    "start": [window.start for window in validation_windows],
    "end": [window.end for window in validation_windows],
    "weight": [window.weight for window in validation_windows],
}))

## Run the causal tuner

In [ ]:
tuning_result = tune_inflow_process_noise(
    storage=tuning_storage,
    discharge=tuning_discharge,
    base_config=base_config,
    candidate_prior_hourly_increment_sd=CANDIDATE_PRIOR_HOURLY_INCREMENT_SD,
    validation_windows=validation_windows,
    settings=settings,
    proposed_configuration_version=PROPOSED_CONFIGURATION_VERSION,
)

print("Selected prior hourly increment SD:", tuning_result.selected_prior_hourly_increment_sd)
print("Selected q_inflow:", tuning_result.selected_q_inflow)
print("Competitive candidates:", tuning_result.competitive_candidates)
print("Selection reason:", tuning_result.selection_reason)
print("Warnings:")
for warning in tuning_result.warnings:
    print(" -", warning)

display(tuning_result.selected_config.metadata)

In [ ]:
print("Candidate summary")
display(tuning_result.candidate_summary)
print("Per-window diagnostics")
display(tuning_result.window_diagnostics)
print("Per-regime diagnostics")
display(tuning_result.regime_diagnostics)
print("Open-loop horizon diagnostics")
display(tuning_result.horizon_diagnostics)
if tuning_result.r_sensitivity is not None:
    print("Measurement-noise sensitivity")
    display(tuning_result.r_sensitivity)

In [ ]:
summary = tuning_result.candidate_summary.sort_values("prior_hourly_increment_sd")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(summary["prior_hourly_increment_sd"], summary["weighted_storage_nlpd"], marker="o")
axes[0].set_xscale("log")
axes[0].set_xlabel("Prior hourly increment SD")
axes[0].set_ylabel("Weighted storage NLPD")
axes[0].set_title("Causal score by candidate")
axes[1].plot(summary["prior_hourly_increment_sd"], summary["joint_nis"], marker="o", label="joint NIS")
axes[1].plot(summary["prior_hourly_increment_sd"], summary["storage_nis"], marker="o", label="storage NIS")
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[1].set_xscale("log")
axes[1].set_xlabel("Prior hourly increment SD")
axes[1].set_ylabel("Normalized innovation squared")
axes[1].legend()
fig.tight_layout()
plt.show()

In [ ]:
selected_batch = get_reservoir_inflow_from_config(
    tuning_storage, tuning_discharge, tuning_result.selected_config
)
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(tuning_storage.index, selected_batch["estimated_inflow"], label="causal estimated inflow")
for window in validation_windows:
    ax.axvspan(window.start, window.end, alpha=0.10, label=window.name)
ax.set_ylabel("Inflow rate")
ax.set_title("Selected configuration: causal inflow estimate")
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()

display(selected_batch[["estimated_inflow", "estimated_inflow_flag"]].tail())

## Optional untouched final-test evaluation

This block is disabled by default. Enable it only when `TEST_START`/`TEST_END` identify a contiguous period that was not used to choose the grid, windows, weights, warm-up, thresholds, or selected candidate. A failed test does not authorize choosing a different candidate from the same test results.

In [ ]:
if RUN_UNTOUCHED_TEST:
    untouched_window = TuningWindow("untouched-test", pd.Timestamp(TEST_START), pd.Timestamp(TEST_END))
    test_result = evaluate_inflow_config(
        storage=full_storage,
        discharge=full_discharge,
        config=tuning_result.selected_config,
        evaluation_window=untouched_window,
        settings=settings,
    )
    display(test_result.candidate_summary)
    display(test_result.window_diagnostics)
    display(test_result.horizon_diagnostics)
    print("Test warnings:")
    for warning in test_result.warnings:
        print(" -", warning)
else:
    print("Untouched-test evaluation is disabled.")

## Interpretation and approval checklist

- Too-small process noise generally produces delayed event response and persistent innovation structure; too-large process noise can produce rough or negative inflow.
- Treat the selected value as a conservative constant-diffusion approximation for the reviewed regimes, not a universal physical constant.
- Review the selected candidate's storage/outflow/conditional-storage NIS, standardized bias, elapsed-time autocorrelation, missing-data coverage, physical behavior, and multi-horizon diagnostics by regime.
- Investigate movement under the optional `R` sensitivity scenarios. That table is diagnostic and cannot replace the primary selection.
- Preserve the proposed configuration metadata and data fingerprint for reproducibility. Persist only after independent engineering approval and an untouched test review.